In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!cp "/content/drive/MyDrive/Forecasting/project1/master_wide_hourly_train_2012_2013.csv" "/content/master_wide_hourly_train_2012_2013.csv"
!cp "/content/drive/MyDrive/Forecasting/project1/master_wide_hourly_validation_2014_01_04.csv" "/content/master_wide_hourly_validation_2014_01_04.csv"
!cp "/content/drive/MyDrive/Forecasting/project1/master_wide_hourly_test_2014_05_12.csv" "/content/master_wide_hourly_test_2014_05_12.csv"
!cp "/content/drive/MyDrive/Forecasting/project1/calendar_features_hourly.csv" "/content/calendar_features_hourly.csv"

In [ ]:
!pip install neuralforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.9/269.9 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 70.8 MB/s eta 0:00:00


In [ ]:
from pathlib import Path

# Input files -- wide-format CSVs (timestamp as index, one column per client)
TRAIN_PATH    = "/content/master_wide_hourly_train_2012_2013.csv"
VAL_PATH      = "/content/master_wide_hourly_validation_2014_01_04.csv"
TEST_PATH     = "/content/master_wide_hourly_test_2014_05_12.csv"
CALENDAR_PATH = "/content/calendar_features_hourly.csv"

# Output
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

# Model hyper-parameters
CHUNK_H       = 720         # forecast horizon per chunk (~1 month); used for training and rolling prediction
INPUT_SIZE    = 672         # look back 672 hours (4 weeks), same as SARIMAX
MAX_STEPS     = 5000
LEARNING_RATE = 1e-3
BATCH_SIZE    = 32
EARLY_STOP    = 10           # patience in validation checks
VAL_CHECK     = 100         # validate every N training steps
RANDOM_SEED   = 42
SCALER_TYPE   = "standard"  # normalise each series independently

In [ ]:
# IMPORTS
import math
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from neuralforecast import NeuralForecast
from neuralforecast.models import iTransformer
from neuralforecast.losses.pytorch import MSE, MAE

In [ ]:
# HELPERS

# Calendar exogenous features -- cyclic encodings + is_weekend
EXOG_COLS = ["hour_sin", "hour_cos", "dow_sin", "dow_cos",
             "is_weekend", "month_sin", "month_cos"]

calendar = pd.read_csv(CALENDAR_PATH, parse_dates=["timestamp"])


def wide_to_long(path):
    """Read a wide-format CSV and melt into NeuralForecast long format.

    Columns in wide CSV: timestamp, MT_124, MT_132, ...
    Returns DataFrame with: unique_id, ds, y
    """
    df = pd.read_csv(path, parse_dates=["timestamp"])
    client_cols = [c for c in df.columns if c != "timestamp"]
    long = df.melt(id_vars="timestamp", value_vars=client_cols,
                   var_name="unique_id", value_name="y")
    long = long.rename(columns={"timestamp": "ds"})
    return long.sort_values(["unique_id", "ds"]).reset_index(drop=True)


def merge_calendar(df, calendar, exog_cols):
    """Left-join calendar features onto the long-format DataFrame."""
    return df.merge(calendar[["timestamp"] + exog_cols].rename(
        columns={"timestamp": "ds"}), on="ds", how="left")


print("Helpers ready.")

Helpers ready.


In [ ]:
# LOAD & PREPARE DATA
train = merge_calendar(wide_to_long(TRAIN_PATH), calendar, EXOG_COLS)
val   = merge_calendar(wide_to_long(VAL_PATH),   calendar, EXOG_COLS)
test  = merge_calendar(wide_to_long(TEST_PATH),  calendar, EXOG_COLS)

N_SERIES = train["unique_id"].nunique()

print(f"Number of clients (n_series): {N_SERIES}")
print(f"train : {train.shape}  {train['ds'].min()} -> {train['ds'].max()}")
print(f"val   : {val.shape}    {val['ds'].min()} -> {val['ds'].max()}")
print(f"test  : {test.shape}   {test['ds'].min()} -> {test['ds'].max()}")
print()
print(train.head())

Number of clients (n_series): 156
train : (2721888, 10)  2012-01-01 00:00:00 -> 2013-12-31 23:00:00
val   : (445536, 10)    2014-01-01 00:00:00 -> 2014-04-30 23:00:00
test  : (913536, 10)   2014-05-01 00:00:00 -> 2014-12-31 23:00:00

                   ds unique_id           y  hour_sin  hour_cos   dow_sin  \
0 2012-01-01 00:00:00    MT_124  108.851670  0.000000  1.000000 -0.781831   
1 2012-01-01 01:00:00    MT_124  123.205750  0.258819  0.965926 -0.781831   
2 2012-01-01 02:00:00    MT_124  114.832535  0.500000  0.866025 -0.781831   
3 2012-01-01 03:00:00    MT_124  101.674644  0.707107  0.707107 -0.781831   
4 2012-01-01 04:00:00    MT_124  105.263150  0.866025  0.500000 -0.781831   

   dow_cos  is_weekend  month_sin  month_cos  
0  0.62349           1        0.5   0.866025  
1  0.62349           1        0.5   0.866025  
2  0.62349           1        0.5   0.866025  
3  0.62349           1        0.5   0.866025  
4  0.62349           1        0.5   0.866025  


In [ ]:
# EVALUATION METRICS
def compute_metrics(df, target_col="y", pred_col="iTransformer"):
    y    = df[target_col].values
    yhat = df[pred_col].values
    mse  = np.mean((y - yhat) ** 2)
    mae  = np.mean(np.abs(y - yhat))
    wape = np.sum(np.abs(y - yhat)) / np.sum(np.abs(y))
    return {"MSE": round(mse, 4), "MAE": round(mae, 4), "WAPE": round(wape, 4)}


def per_client_metrics(df, target_col="y", pred_col="iTransformer"):
    rows = []
    for uid, grp in df.groupby("unique_id"):
        rows.append({"client_id": uid, **compute_metrics(grp, target_col, pred_col)})
    rows.append({"client_id": "OVERALL", **compute_metrics(df, target_col, pred_col)})
    return pd.DataFrame(rows)

In [ ]:
val_h = val["ds"].nunique()
print(f"Validation horizon (val_h): {val_h} hours")
print(f"Chunk horizon (CHUNK_H):    {CHUNK_H} hours (~1 month)")

# Fit on train + val combined, using val as internal validation for early stopping
train_val_nf = pd.concat([train, val], ignore_index=True).sort_values(
    ["unique_id", "ds"]).reset_index(drop=True)

model = iTransformer(
    h=CHUNK_H,
    input_size=INPUT_SIZE,
    n_series=N_SERIES,
    hidden_size=512,
    n_heads=8,
    e_layers=2,
    d_layers=1,
    d_ff=2048,
    factor=1,
    dropout=0.1,
    use_norm=True,
    loss=MSE(),
    valid_loss=MAE(),
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    early_stop_patience_steps=EARLY_STOP,
    val_check_steps=VAL_CHECK,
    batch_size=BATCH_SIZE,
    scaler_type=SCALER_TYPE,
    random_seed=RANDOM_SEED,
)

nf = NeuralForecast(models=[model], freq="h")
nf.fit(df=train_val_nf, val_size=val_h)  # val_h >= CHUNK_H, so no constraint issue

nf.save(str(MODEL_DIR / "itransformer_val"))
print("Training complete. Model saved -> models/itransformer_val/")

Validation horizon (val_h): 2856 hours
Chunk horizon (CHUNK_H):    720 hours (~1 month)


INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type                   ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss          │ MSE                    │      0 │ train │     0 │
│ 1 │ valid_loss    │ MAE                    │      0 │ train │     0 │
│ 2 │ padder_train  │ ConstantPad1d          │      0 │ train │     0 │
│ 3 │ scaler        │ TemporalNorm           │      0 │ train │     0 │
│ 4 │ enc_embedding │ DataEmbedding_inverted │  344 K │ train │     0 │
│ 5 │ encoder       │ TransEncoder           │  6.3 M │ train │     0 │
│ 6 │ projector     │ Linear                 │  369 K │ train │     0 │
└───┴───────────────┴────────────────────────┴────────┴───────┴───────┘

Trainable params: 7.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 7.0 M                                                                                                
Total estimated model params size (MB): 28                                                                         
Modules in train mode: 37                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Training complete. Model saved -> models/itransformer_val/


In [ ]:
# VALIDATION FORECAST — rolling prediction with CHUNK_H-step chunks

val_dates = sorted(val["ds"].unique())
history_val = train.copy()
val_all_preds = []

n_chunks_val = math.ceil(val_h / CHUNK_H)
print(f"Val rolling prediction: {n_chunks_val} chunk(s) of up to {CHUNK_H} hours each")

for i in range(n_chunks_val):
    chunk_dates = set(val_dates[i * CHUNK_H : (i + 1) * CHUNK_H])

    preds = nf.predict(df=history_val).reset_index()
    preds["ds"] = pd.to_datetime(preds["ds"])

    chunk_preds = preds[preds["ds"].isin(chunk_dates)]
    val_all_preds.append(chunk_preds)

    pred_rows = chunk_preds[["unique_id", "ds", "iTransformer"]].rename(
        columns={"iTransformer": "y"}
    )
    pred_rows = merge_calendar(pred_rows, calendar, EXOG_COLS)
    history_val = pd.concat([history_val, pred_rows], ignore_index=True).sort_values(
        ["unique_id", "ds"]).reset_index(drop=True)

val_preds = pd.concat(val_all_preds, ignore_index=True)
val_preds["ds"] = pd.to_datetime(val_preds["ds"])
val["ds"] = pd.to_datetime(val["ds"])

val_eval = val[["unique_id", "ds", "y"]].merge(val_preds, on=["unique_id", "ds"])
print(val_eval.head())

Val rolling prediction: 4 chunk(s) of up to 720 hours each


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

  unique_id                  ds          y  index  iTransformer
0    MT_124 2014-01-01 00:00:00  95.693780      0    338.241699
1    MT_124 2014-01-01 01:00:00  83.732056      1    284.217102
2    MT_124 2014-01-01 02:00:00  82.535890      2    219.476242
3    MT_124 2014-01-01 03:00:00  82.535890      3    152.609161
4    MT_124 2014-01-01 04:00:00  63.397133      4    117.452347


In [ ]:
# TRAINING METRICS (In-Sample via Cross-Validation) + VALIDATION METRICS (Out-of-Sample)

# 1. Training Metrics — use cross_validation to get "in-sample" predictions
#    predict_insample() is NOT supported for multivariate models like iTransformer.
#    Instead, we run nf.predict() on a truncated training set as a proxy.

# Use only the last portion of training data as "in-sample" check:
# Feed all but the last val_h rows of training data, then predict the last val_h rows.
train_h = val_h  # use same horizon for comparability
cutoff = train.groupby("unique_id")["ds"].apply(
    lambda s: s.sort_values().iloc[-train_h]
).min()

train_hist = train[train["ds"] < cutoff].reset_index(drop=True)
train_future = train[train["ds"] >= cutoff].reset_index(drop=True)

train_preds = nf.predict(df=train_hist)
train_preds = train_preds.reset_index()
train_preds["ds"] = pd.to_datetime(train_preds["ds"])
train_future["ds"] = pd.to_datetime(train_future["ds"])

train_eval = train_future[["unique_id", "ds", "y"]].merge(
    train_preds, on=["unique_id", "ds"], how="inner")

train_metrics = per_client_metrics(train_eval.dropna())
print("-- Training Metrics (In-Sample proxy via predict on held-out tail) --")
print(train_metrics.head(5).to_string(index=False))
print("...")
print(train_metrics.tail(1).to_string(index=False))
print()

# 2. Validation Metrics (Out-of-Sample)
val_metrics = per_client_metrics(val_eval)
print("-- Validation Metrics (Out-of-Sample) --")
print(val_metrics.head(5).to_string(index=False))
print("...")
print(val_metrics.tail(1).to_string(index=False))

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

-- Training Metrics (In-Sample proxy via predict on held-out tail) --
client_id       MSE     MAE   WAPE
   MT_124 6199.2775 64.0420 0.2217
   MT_132  302.5179 13.0273 0.9774
   MT_156  883.9421 23.0828 0.2279
   MT_158 2627.2774 43.2639 0.6712
   MT_159 1525.6542 31.4799 0.6136
...
client_id         MSE     MAE   WAPE
  OVERALL 145887.5842 80.4921 0.1099

-- Validation Metrics (Out-of-Sample) --
client_id       MSE     MAE   WAPE
   MT_124 4154.7859 52.8770 0.1951
   MT_132  772.7024 21.8893 1.2822
   MT_156  822.5328 24.0604 0.2943
   MT_158 2440.8162 39.7361 0.5387
   MT_159 2249.2349 38.0010 0.7497
...
client_id        MSE     MAE   WAPE
  OVERALL 60358.2331 68.9378 0.1123


In [ ]:
# TEST EVALUATION
# Rolling prediction using nf (h=CHUNK_H): each iteration appends predicted values
# as history before predicting the next chunk — consistent with SARIMAX evaluation.

test_h = test["ds"].nunique()
print(f"Test horizon (test_h): {test_h} hours")

test_dates = sorted(test["ds"].unique())
history = train_val_nf.copy()
all_preds = []

n_chunks = math.ceil(test_h / CHUNK_H)
print(f"Rolling prediction: {n_chunks} chunk(s) of up to {CHUNK_H} hours each")

for i in range(n_chunks):
    chunk_dates = set(test_dates[i * CHUNK_H : (i + 1) * CHUNK_H])

    preds = nf.predict(df=history).reset_index()
    preds["ds"] = pd.to_datetime(preds["ds"])

    chunk_preds = preds[preds["ds"].isin(chunk_dates)]
    all_preds.append(chunk_preds)

    pred_rows = chunk_preds[["unique_id", "ds", "iTransformer"]].rename(
        columns={"iTransformer": "y"}
    )
    pred_rows = merge_calendar(pred_rows, calendar, EXOG_COLS)
    history = pd.concat([history, pred_rows], ignore_index=True).sort_values(
        ["unique_id", "ds"]).reset_index(drop=True)

test_preds_all = pd.concat(all_preds, ignore_index=True)
test_eval = test[["unique_id", "ds", "y"]].merge(test_preds_all, on=["unique_id", "ds"])

test_metrics = per_client_metrics(test_eval)
print("-- Test Metrics --")
print(test_metrics.to_string(index=False))

Test horizon (test_h): 5856 hours
Rolling prediction: 9 chunk(s) of up to 720 hours each


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

...

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()

-- Test Metrics --
client_id          MSE       MAE   WAPE
   MT_124 6.821536e+03   65.6530 0.2184
   MT_132 3.518982e+02   16.0672 1.0631
   MT_156 6.388328e+02   20.6503 0.2325
   MT_158 1.682637e+03   34.3128 0.4388
   MT_159 1.093843e+03   26.4295 0.6197
   MT_161 1.984013e+05  395.2182 0.2522
   MT_162 1.685147e+04  108.2342 0.3993
   MT_163 1.572287e+05  309.4922 0.1238
   MT_166 5.167861e+04  179.4921 0.1390
   MT_168 7.477850e+02   21.9806 0.1642
   MT_169 7.818280e+01    7.2607 0.1723
   MT_171 2.126811e+03   38.7188 0.1640
   MT_172 8.075754e+02   23.3549 0.1912
   MT_174 1.632311e+03   31.2311 0.1900
   MT_175 1.958181e+03   36.3659 0.2285
   MT_176 5.057146e+02   18.3204 0.2123
   MT_180 1.339341e+03   27.9304 0.1867
   MT_182 1.332872e+02    9.5726 0.1900
   MT_183 3.280204e+02   13.4852 0.1741
   MT_187 6.323580e+02   20.8098 0.2238
   MT_188 7.902440e+01    7.2490 0.1709
   MT_189 6.114281e+03   60.0642 0.1501
   MT_190 4.265182e+03   49.4071 0.1402
   MT_191 1.294205e+0

In [ ]:
import shutil
from google.colab import files

# Zip the directories
shutil.make_archive("/content/itransformer_val", 'zip', "/content/models/itransformer_val")

print("Zipping complete.")

Zipping complete.


In [ ]:
# Download the zip files
files.download("/content/itransformer_val.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>